In [3]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

## Helper functions

In [2]:
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## Test Implementation

### Image analysis

In [ ]:
# Fire risk assessment prompt

prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

#### Read image data, feed into LLM

In [5]:
with open("./assets/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": image_bytes,
            }
        },
        {
            "type": "text",
            "text": prompt,
        }
    ],
)

In [6]:
messages

[{'role': 'user',
  'content': [{'type': 'image',
    'source': {'type': 'base64',
     'media_type': 'image/png',
     'data': '...'}},
   {'type': 'text',
    'text': '\nAnalyze the attached satellite image of a property with these specific steps:\n\n1. Residence identification: Locate the primary residence on the property by looking for:\n   - The largest roofed structure \n   - Typical residential features (driveway connection, regular geometry)\n   - Distinction from other structures (garages, sheds, pools)\n   Describe the residence\'s location relative to property boundaries and other features.\n\n2. Tree overhang analysis: Examine all trees near the primary residence:\n   - Identify any trees whose canopy extends directly over any portion of the roof\n   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)\n   - Note particularly dense areas of overhang\n\n3. Fire risk assessment: For any overhanging trees, evaluate:\n   - Potential

In [7]:
chat_response = chat(messages)

In [8]:
print(text_from_message(chat_response))

# Property Fire Risk Analysis

**1. Residence Identification:**
The primary residence is a light-colored (gray/tan) roofed structure located in the center of the image, characterized by an irregular or L-shaped footprint with what appears to be multiple roof sections, surrounded by dense forest vegetation on all sides with minimal visible cleared area.

**2. Tree Overhang Analysis:**
Multiple large tree canopies directly overhang significant portions of the residence's roof, with an estimated 50-75% of the roof area covered by overhanging branches from mature trees positioned immediately adjacent to and surrounding the structure.

**3. Fire Risk Assessment:**
The overhanging trees create multiple ember catch points across the roof surface, establish direct fuel pathways from the surrounding forest canopy to the structure, and form continuous vegetation bridges that would allow fire to readily transfer from wildland fuels to the building.

**4. Defensible Space Identification:**
The pro

### PDF Support

In [9]:
with open("./assets/earth.pdf", "rb") as f:
    doc_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": doc_bytes,
            }
        },
        {
            "type": "text",
            "text": "Summarize the key points from this PDF in less than 250 words.",
        }
    ],
)

In [10]:
chat_response = chat(messages)
print(text_from_message(chat_response))

# Earth: Key Facts Summary

**Basic Overview:**
Earth is the third planet from the Sun and the only known astronomical object harboring life. It's an ocean world with 70.8% of its surface covered by water, with the remaining 29.2% being land.

**Physical Characteristics:**
- Mean radius: 6,371 km
- Mass: 5.97 × 10²⁴ kg
- Densest planet in the Solar System
- Circumference: approximately 40,000 km
- Surface gravity: 9.81 m/s²

**Orbital Properties:**
- Distance from Sun: ~149.6 million km (8 light-minutes)
- Orbital period: 365.26 days
- Rotation period: 23 hours 56 minutes
- Axial tilt: 23.44°, producing seasons
- One natural satellite: the Moon

**Atmosphere & Climate:**
- Composition: 78% nitrogen, 21% oxygen, plus water vapor and trace gases
- Average surface temperature: 14.76°C (58.57°F)
- Dynamic atmosphere with greenhouse gases (CO₂, water vapor) that maintain liquid water
- Protective magnetosphere deflects solar wind

**Formation & History:**
- Formed approximately 4.54 billion